# Lee2019 Fine-Tuning with SignalJEPA Downstream Variants

Downstream fine-tuning on **Lee2019 Dataset**.

- Dataset: Lee2019 (MOABB)
- EEG channels only; bandpass 0.5-40 Hz; resample to 128 Hz
- 5-fold stratified within-subject cross-validation
- Model: configurable via `CONFIG["model_name"]`
- Pretrained weights: Hugging Face 16s-60 checkpoint
- Fine-tuning strategy: configurable via `CONFIG["strategy"]` (`new` or `full`)
- Training: Braindecode EEGClassifier with paper-aligned early stopping

# 1. Setup

## 1.1. Import Libraries

In [ ]:
import os
import random
import sys
from pathlib import Path
import platform
import json
import hashlib
from datetime import datetime

import pandas as pd
import numpy as np
from numpy import multiply

import torch
from torch.utils.data import Subset

from braindecode import EEGClassifier
from braindecode.models import SignalJEPA_PreLocal, SignalJEPA_PostLocal, SignalJEPA_Contextual
from braindecode.datasets import MOABBDataset
from braindecode.preprocessing import (
    Preprocessor,
    preprocess,
    create_windows_from_events,
)

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from skorch.callbacks import EarlyStopping
from skorch.helper import predefined_split

import builtins

from moabb.datasets import Lee2019_SSVEP, Lee2019_ERP, Lee2019_MI

import mne
mne.set_log_level("WARNING")

print("All imports loaded successfully.")

## 1.2. Runtime & Path Validation

In [ ]:
print("Runtime Environment:")
print(f"  - Python: {sys.version}")
print(f"  - Platform: {platform.platform()}")

WORKING_DIR = Path.cwd().parent
print(f"\nWorking directory: {WORKING_DIR}")

# 2. Configuration

## 2.1. Config

In [ ]:
CONFIG = {
    # Paths
    "artifact_dir": str(WORKING_DIR / "artifacts" / "lee-2019-fine-tuning"),

    # Reproducibility
    "seed": None,
    "set_seed": False,

    # Preprocessing (paper-aligned)
    "sfreq": 128,
    "bandpass_low": 0.5,
    "bandpass_high": 40.0,

    # Preprocessing (implementation defaults, not paper claims)
    "set_eeg_reference": True,
    "set_standard_montage": True,
    "convert_to_uV": True,
    "learning_rate": None, # None = default skorch learning rate 0.01

    # Subjects (None = auto-select last 7, or specify list of subject IDs)
    "subjects_to_use": [51],

    # Paradigm settings
    "paradigm": "SSVEP", # 'SSVEP', 'MI', 'ERP'

    # Model settings (paper-aligned)
    "model_name": "SignalJEPA_PreLocal", # SignalJEPA_PreLocal, SignalJEPA_PostLocal, SignalJEPA_Contextual

    # Schema settings (paper-aligned)
    "strategy": "new", # new, full
    "warmup_epochs": 10, # paper-aligned for 'full' strategy

    # Cross-validation (paper-aligned structure)
    "cv_folds": 5,
    "val_split": 0.2,

    # Training (implementation defaults, not paper claims)
    "batch_size": 32,
    "n_epochs": 5000,
    "early_stopping_patience": 50,

    # Pretrained weights (Hugging Face 16s-60 checkpoint)
    "pretrained_url": "https://huggingface.co/braindecode/SignalJEPA/resolve/main/signal-jepa_16s-60_adeuwv4s.pth",
}

# Output verbosity flags (set to True for more detailed output)
LOG_COMPACT = False
PRINT_MODEL_SUMMARY = True
PRINT_FREEZE_DETAILS = True
PRINT_FOLD_CLASS_COUNTS = True


In [ ]:
PARADIGM_CONFIGS = {
    "SSVEP": {
        "n_classes": 4,
        "base_trial_duration_s": 4.0,
        "requested_trial_duration_s": 4.19,
        "window_samples_size": 536,
    },
    "MI": {
        "n_classes": 2,
        "base_trial_duration_s": 4.0,
        "requested_trial_duration_s": 4.19,
        "window_samples_size": 536,
    },
    "ERP": {
        "n_classes": 2,
        "base_trial_duration_s": 1.0,
        "requested_trial_duration_s": 1.19,
        "window_samples_size": 152,
    },
}

## 2.2. Create Artifact Directory

In [ ]:
def create_run_id():
    # Generate unique run ID from timestamp + config hash.
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    config_str = json.dumps(CONFIG, sort_keys=True, default=str)
    config_hash = hashlib.md5(config_str.encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"

In [ ]:
RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / CONFIG["paradigm"] / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

## 2.3. Initialize Logger

In [ ]:
LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1)

def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop("sep", " ")
    end = kwargs.pop("end", "\n")
    flush = kwargs.pop("flush", False)
    file = kwargs.pop("file", None)

    message = sep.join(str(arg) for arg in args)

    # Preserve visual spacing for prints like print("\n...")
    leading_newlines = len(message) - len(message.lstrip("\n"))
    message_body = message[leading_newlines:]

    def _write_target(text):
        if file is None:
            sys.__stdout__.write(text) # type: ignore
            if flush:
                sys.__stdout__.flush() # type: ignore
        else:
            file.write(text)
            if flush and hasattr(file, "flush"):
                file.flush()

    # Apply leading blank lines first without timestamps
    if leading_newlines > 0:
        blanks = "\n" * leading_newlines
        _write_target(blanks)
        _LOG_FILE_HANDLE.write(blanks)

    if message_body:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        stamped = f"[{ts}] {message_body}"
        _write_target(stamped + end)
        _LOG_FILE_HANDLE.write(stamped + end)
    else:
        # If only newlines were printed, preserve end behavior without adding a timestamp
        _write_target(end)
        _LOG_FILE_HANDLE.write(end)

    if flush:
        _LOG_FILE_HANDLE.flush()

builtins.print = _timestamped_print

## 2.4. Device Configuration

In [ ]:
def resolve_device():
    """Resolve the compute device."""
    # prefer MPS > CUDA > CPU
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


DEVICE = resolve_device()
print(f"Using device: {DEVICE}")

## 2.5. Deterministic Seeding

In [ ]:
def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

    torch.use_deterministic_algorithms(True, warn_only=True)

def seed_worker(worker_id):
    _ = worker_id
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

BASE_SEED = int(CONFIG["seed"]) if CONFIG["seed"] is not None else None

if CONFIG["set_seed"]:
    if BASE_SEED is None:
        raise ValueError("Seed control is enabled but CONFIG['seed'] is None. Please specify a seed.")

    seed_everything(BASE_SEED)
    print(f"Seed control enabled: {CONFIG['set_seed']}")
    print(f"Base seed: {BASE_SEED}")
    print(f"Seed initialized: {BASE_SEED}")
else:
    print("Seed control disabled (CONFIG['set_seed'] = False)")

## 2.6. Save Configuration

In [ ]:
print("=" * 70)
print("CONFIGURATION")
print("=" * 70)
for key in sorted(CONFIG.keys()):
    print(f"  {key}: {CONFIG[key]}")
print("=" * 70)

config_path = ARTIFACT_DIR / "config.json"
with open(config_path, 'w') as f:
    json.dump(CONFIG, f, indent=2, default=str)
print(f"Config saved to: {config_path}")

# 3. Prepare Data

## 3.1. Derived Constants

In [ ]:
def current_paradigm_config(paradigm, paradigm_configs):
    if paradigm not in ["SSVEP", "ERP", "MI"]:
        raise ValueError(f"Unsupported paradigm: {paradigm}. Must be one of 'SSVEP', 'ERP', 'MI'.")

    return (
        paradigm_configs[paradigm]["n_classes"],
        paradigm_configs[paradigm]["base_trial_duration_s"],
        paradigm_configs[paradigm]["requested_trial_duration_s"],
        paradigm_configs[paradigm]["window_samples_size"],
    )

In [ ]:
def compute_epoch_window(sfreq, base_trial_duration_s, target_trial_duration_s):
    """Compute the exact downstream window and stop offset from CONFIG."""
    sfreq = float(sfreq)
    target_trial_duration_s = float(target_trial_duration_s)

    window_size_samples = round(target_trial_duration_s * sfreq)

    base_trial_duration_s = float(base_trial_duration_s)
    base_trial_samples = round(base_trial_duration_s * sfreq)
    trial_stop_offset_samples = window_size_samples - base_trial_samples
    effective_duration = window_size_samples / sfreq

    print("Epoch window configuration:")
    print(f"  Target sfreq:                {sfreq:.0f} Hz")
    print(f"  Target duration:             {target_trial_duration_s:.6f} s")
    print(f"  Window samples (rounded):    {window_size_samples}")
    print(f"  Base trial samples ({base_trial_duration_s:.1f} s):  {base_trial_samples}")
    print(f"  Trial stop offset samples:   {trial_stop_offset_samples}")
    print(f"  Effective window duration:   {effective_duration:.6f} s")

    return window_size_samples, base_trial_samples, trial_stop_offset_samples

In [ ]:
TARGET_N_CLASSES, BASE_TRIAL_DURATION_S, TARGET_TRIAL_DURATION_S, EXPECTED_WINDOW_SAMPLE_SIZE = current_paradigm_config(CONFIG["paradigm"], PARADIGM_CONFIGS)

WINDOW_SAMPLES, BASE_TRIAL_SAMPLES, TRIAL_STOP_OFFSET_SAMPLES = compute_epoch_window(
    sfreq=CONFIG["sfreq"],
    base_trial_duration_s=BASE_TRIAL_DURATION_S,
    target_trial_duration_s=TARGET_TRIAL_DURATION_S,
)

## 3.2. Subject Selection

In [ ]:
def select_last_subjects(dataset, n_last=7):
    """Select the last n subjects required for downstream evaluation."""

    print("Dataset code:", dataset.code)
    print("Subjects Total:", len(dataset.subject_list))
    print("Events:", dataset.event_id)
    print("Interval:", dataset.interval)
    print("Paradigm:", dataset.paradigm)

    subjects = sorted(int(s) for s in dataset.subject_list)

    if len(subjects) < n_last:
        raise RuntimeError(f"Dataset has {len(subjects)} subjects, need at least {n_last}.")
    return subjects[-n_last:]

In [ ]:
if CONFIG["paradigm"] == "SSVEP":
    LEE_DATASET = Lee2019_SSVEP() # type: ignore
elif CONFIG["paradigm"] == "MI":
    LEE_DATASET = Lee2019_MI() # type: ignore
elif CONFIG["paradigm"] == "ERP":
    LEE_DATASET = Lee2019_ERP() # type: ignore

if CONFIG["subjects_to_use"] is not None:
    SUBJECTS = CONFIG["subjects_to_use"]
    print(f"Using specified subjects from config: {SUBJECTS}")
else:
    SUBJECTS = select_last_subjects(LEE_DATASET, n_last=7)

print(f"Selected subjects: {SUBJECTS}")
print(f"Expected evaluation folds: {len(SUBJECTS) * CONFIG['cv_folds']}")

DATASET_NAME = f"Lee2019_{CONFIG['paradigm']}"
print(f"\nBuilding MOABBDataset for {DATASET_NAME}...")
DATASET = MOABBDataset(
    dataset_name=DATASET_NAME,
    subject_ids=SUBJECTS,
    dataset_kwargs={"train_run": True, "test_run": False},
)
print(f"MOABBDataset recordings loaded: {len(DATASET.datasets)}")

# event mapping diagnostics
event_id = dict(sorted(LEE_DATASET.event_id.items(), key=lambda kv: kv[1]))
label_map = {code: idx for idx, (_, code) in enumerate(event_id.items())}
first_ann = sorted(set(DATASET.datasets[0].raw.annotations.description))
print(f"Discovered annotation descriptions (first recording): {first_ann}")
print(f"Dataset event_id mapping: {event_id}")
print(f"Label map (event_code -> class_index): {label_map}")

## 3.3. Filter and Resample Data

In [ ]:
def get_preprocessors(sfreq, bandpass_low, bandpass_high, set_eeg_reference, set_standard_montage, convert_to_uV):
    """Build Braindecode preprocessors in memory-efficient order.

    Preprocessing steps:
      1. Pick EEG channels.
      2. Attach a standard montage (needed for contextual spatial positional encoding). [OPTIONAL]
      3. Set EEG reference to average.
      4. Convert V -> uV (scale raw voltage to microvolts). [OPTIONAL]
      5. Bandpass filter.
      6. Resample to the target sampling frequency.
    """

    preprocessors = [Preprocessor("pick", picks="eeg")]

    if set_standard_montage:
        preprocessors.append(
            Preprocessor(
                "set_montage",
                montage="standard_1005",
                on_missing="ignore",
                match_case=False,
            )
        )

    if set_eeg_reference:
        preprocessors.append(Preprocessor("set_eeg_reference", ref_channels="average"))

    if convert_to_uV:
        preprocessors.append(Preprocessor(lambda data: multiply(data, 1e6)))

    preprocessors.append(Preprocessor("filter", l_freq=bandpass_low, h_freq=bandpass_high, method="iir"))
    preprocessors.append(Preprocessor("resample", sfreq=sfreq))

    return preprocessors

## 3.4. Channel Info and Validation

In [ ]:
def get_channel_info_from_one_recording(dataset):
    """Extract channel metadata from the first preprocessed recording."""
    first_raw = dataset.datasets[0].raw
    return first_raw.info["chs"], len(first_raw.ch_names), list(first_raw.ch_names)

In [ ]:
def validate_channel_consistency(dataset):
    """Validate all recordings share channel count/order after preprocessing."""
    checked_runs = 0
    unique_channel_counts = set()
    reference_names = None
    inconsistent_runs = []

    for ds in dataset.datasets:
        checked_runs += 1
        raw = ds.raw
        ch_names = tuple(raw.ch_names)
        unique_channel_counts.add(len(ch_names))

        if reference_names is None:
            reference_names = ch_names
            continue

        if ch_names != reference_names:
            inconsistent_runs.append(
                {
                    "run": str(ds.description.to_dict()),
                    "n_channels": len(ch_names),
                    "first_channels": list(ch_names[:10]),
                }
            )

    channel_order_consistent = len(inconsistent_runs) == 0
    print("Channel consistency summary:")
    print(f"  Checked runs: {checked_runs}")
    print(f"  Unique channel counts: {sorted(unique_channel_counts)}")
    print(f"  Channel order consistent: {channel_order_consistent}")

    if inconsistent_runs:
        print(f"  Inconsistent examples: {inconsistent_runs[:2]}")

    return {
        "checked_runs": checked_runs,
        "unique_channel_counts": sorted(unique_channel_counts),
        "channel_order_consistent": channel_order_consistent,
    }

In [ ]:
print("Applying Braindecode preprocessors to MOABBDataset...")
preprocessors = get_preprocessors(
    sfreq=CONFIG["sfreq"],
    bandpass_low=CONFIG["bandpass_low"],
    bandpass_high=CONFIG["bandpass_high"],
    set_eeg_reference=CONFIG["set_eeg_reference"],
    set_standard_montage=CONFIG["set_standard_montage"],
    convert_to_uV=CONFIG["convert_to_uV"],
)

# n_jobs=1 is intentional: each parallel worker would load a full copy of the
# dataset into memory, causing large RAM spikes on local machines.
# Sequential processing (n_jobs=1) is slower but keeps peak memory manageable.
preprocess(DATASET, preprocessors, n_jobs=1)
print("Preprocessing complete.")

# Post-preprocessing validation
print("\nPost-preprocessing validation")
print(f"  Recordings in DATASET: {len(DATASET.datasets)}")
subject_ids = sorted(set(str(ds.description["subject"]) for ds in DATASET.datasets))
print(f"  Unique subject IDs:    {subject_ids}")
first_raw = DATASET.datasets[0].raw
print(f"  sfreq (first rec):     {first_raw.info['sfreq']} Hz")
print(f"  EEG channel count:     {len(first_raw.ch_names)}")
print(f"  First 10 channels:     {list(first_raw.ch_names[:10])}")

In [ ]:
# validate_channel_consistency(DATASET)
CHS_INFO, N_CHANNELS, CH_NAMES = get_channel_info_from_one_recording(DATASET)

print(f"EEG channel #: {N_CHANNELS}")
print(f"First 10 EEG channels: {CH_NAMES[:10]}")

In [ ]:
# Diagnostic: verify channel-location variability for contextual positional encoding.
locs = np.asarray([ch["loc"][:3] for ch in CHS_INFO], dtype=float)
print("Channel-location ranges (x, y, z):")
for i, axis in enumerate(["x", "y", "z"]):
    print(f"  {axis}: min={locs[:, i].min():.6f}, max={locs[:, i].max():.6f}, span={(locs[:, i].max() - locs[:, i].min()):.6f}")

## 3.5. Subject Preprocessing

In [ ]:
def build_windows_dataset(dataset, window_size_samples, trial_stop_offset_samples):
    """Create event-based windows from preprocessed MOABBDataset."""
    trial_start_offset_samples = 0

    print(f"Requested window samples: {window_size_samples}")
    print(f"Base trial samples:       {BASE_TRIAL_SAMPLES}")
    print(f"Stop offset samples:      {trial_stop_offset_samples}")

    windows_dataset = create_windows_from_events(
        dataset,
        trial_start_offset_samples=trial_start_offset_samples,
        trial_stop_offset_samples=trial_stop_offset_samples,
        window_size_samples=window_size_samples,
        window_stride_samples=window_size_samples,
        drop_last_window=False,
        preload=True,
    )

    return windows_dataset

In [ ]:
def get_targets_from_dataset(dataset):
    """Extract integer labels from a Braindecode dataset/subset."""
    return np.asarray([int(dataset[i][1]) for i in range(len(dataset))], dtype=np.int64)

In [ ]:
def get_subject_windows(windows_dataset, subject_ids):
    """Split windows by subject and ensure all requested subjects are available."""
    split_by_subject = windows_dataset.split("subject")
    subject_windows = {}

    for sid in subject_ids:
        key = str(sid)
        if key in split_by_subject:
            subject_windows[sid] = split_by_subject[key]
        elif sid in split_by_subject:
            subject_windows[sid] = split_by_subject[sid]
        else:
            raise RuntimeError(f"Subject {sid} not found in windows split keys: {list(split_by_subject.keys())[:10]}")

    return subject_windows

In [ ]:
# raw = DATASET.datasets[0].raw
# data = raw.get_data()
# print(f"raw mean_abs={np.mean(np.abs(data)):.6e}")
# print(f"raw std={np.std(data):.6e}")
# print("raw first5 channel std=", np.std(data, axis=1)[:5])

WINDOWS_DATASET = build_windows_dataset(
    DATASET,
    window_size_samples=WINDOW_SAMPLES,
    trial_stop_offset_samples=TRIAL_STOP_OFFSET_SAMPLES,
)

# Global label diagnostics
ALL_LABELS = get_targets_from_dataset(WINDOWS_DATASET)
all_counts = np.bincount(ALL_LABELS, minlength=TARGET_N_CLASSES)
print(f"Global class counts: {all_counts.tolist()}")
print(f"Chance level ({TARGET_N_CLASSES}-class): {1.0 / TARGET_N_CLASSES:.2f}")

# Window/sample diagnostics
sample0 = WINDOWS_DATASET[0]
x0 = sample0[0]
y0 = sample0[1]
print(f"One window shape: {tuple(x0.shape)}")
print(f"One window label: {int(y0)}")
print(f"Window sample length expected={WINDOW_SAMPLES}, got={x0.shape[-1]}")

SUBJECT_WINDOWS = get_subject_windows(WINDOWS_DATASET, SUBJECTS)
print(f"Subjects in window split: {list(SUBJECT_WINDOWS.keys())}")

In [ ]:
def summarize_subject_windows(subject_windows, n_classes):
    """Summarize per-subject window counts and class balance."""
    rows = []
    for subject_id in sorted(subject_windows):
        ds = subject_windows[subject_id]
        y = get_targets_from_dataset(ds)
        counts = np.bincount(y, minlength=n_classes)
        sample = ds[0]
        x = sample[0]

        print(
            f"  Subject {subject_id}: n_windows={len(ds)}, "
            f"window_shape={tuple(x.shape)}, class_counts={counts.tolist()}"
        )
        rows.append(
            {
                "subject_id": int(subject_id),
                "n_windows": int(len(ds)),
                "n_channels": int(x.shape[0]),
                "n_times": int(x.shape[1]),
                "class_counts": counts.tolist(),
            }
        )

    return pd.DataFrame(rows)

In [ ]:
print("Summarizing per-subject windows...")
SUBJECT_TRIALS_SUMMARY = summarize_subject_windows(SUBJECT_WINDOWS, TARGET_N_CLASSES)
print(SUBJECT_TRIALS_SUMMARY)

# Keep compatibility with downstream reporting code paths.
SUBJECT_TRIALS = {sid: (None, get_targets_from_dataset(ds)) for sid, ds in SUBJECT_WINDOWS.items()}

## 3.6. Validation and Summary

In [ ]:
def inspect_preprocessing_sanity(windows_dataset):
    """Quick sanity checks on one window after preprocessing."""
    sample = windows_dataset[0]
    x = sample[0]
    y = sample[1]
    x_np = np.asarray(x)

    print("Preprocessing sanity checks:")
    print(f"  One sample label: {int(y)}")
    print(f"  One sample shape: {tuple(x_np.shape)}")
    print(f"  Mean(abs(signal)): {float(np.mean(np.abs(x_np))):.6e}")
    print(f"  Mean(signal):      {float(np.mean(x_np)):.6e}")
    print(f"  Std(signal):       {float(np.std(x_np)):.6e}")

In [ ]:
print("Window validation summary:")

assert len(WINDOWS_DATASET) > 0, "No windows created."
sample0 = WINDOWS_DATASET[0]
x0 = sample0[0]
assert x0.shape[-1] == WINDOW_SAMPLES, (
    f"Window length mismatch: got {x0.shape[-1]} expected {WINDOW_SAMPLES}"
)
assert WINDOW_SAMPLES == EXPECTED_WINDOW_SAMPLE_SIZE, f"Expected {EXPECTED_WINDOW_SAMPLE_SIZE} samples, got {WINDOW_SAMPLES}"

all_labels = get_targets_from_dataset(WINDOWS_DATASET)
assert np.isin(all_labels, np.arange(TARGET_N_CLASSES)).all(), (
    f"Invalid labels found: {np.unique(all_labels)}"
)

for sid, ds in SUBJECT_WINDOWS.items():
    y_sid = get_targets_from_dataset(ds)
    counts = np.bincount(y_sid, minlength=TARGET_N_CLASSES)
    print(f"  Subject {sid}: n_windows={len(ds)} class_counts={counts.tolist()}")

effective_duration = WINDOW_SAMPLES / CONFIG["sfreq"]

print(f"\nValidated sampling rate target: {CONFIG['sfreq']} Hz")
print(f"Validated window sample count:   {WINDOW_SAMPLES}")
print(f"Validated window duration:       {effective_duration:.4f} s")
print(f"Validated channel count:         {N_CHANNELS}")
print(f"Total windows retained:          {len(WINDOWS_DATASET)}")

inspect_preprocessing_sanity(WINDOWS_DATASET)

# 4. Model

## 4.1. Build Model

In [ ]:
MODEL_REGISTRY = {
    "SignalJEPA_PreLocal": SignalJEPA_PreLocal,
    "SignalJEPA_PostLocal": SignalJEPA_PostLocal,
    "SignalJEPA_Contextual": SignalJEPA_Contextual,
}

def get_model_class(model_name):
    """Resolve downstream model class from config name with validation."""
    if model_name not in MODEL_REGISTRY:
        supported = ", ".join(MODEL_REGISTRY.keys())
        raise ValueError(f"Unsupported model_name '{model_name}'. Supported: {supported}")
    return MODEL_REGISTRY[model_name]

def _coord_span(chs_info, start, stop):
    coords = np.asarray([np.asarray(ch["loc"])[start:stop] for ch in chs_info], dtype=float)
    return float(np.nanmax(coords) - np.nanmin(coords))

def adapt_chs_info_for_contextual(chs_info):
    """Ensure contextual positional encoding sees non-degenerate channel coordinates."""
    span_03 = _coord_span(chs_info, 0, 3)
    span_36 = _coord_span(chs_info, 3, 6)

    if span_36 > 0.0 or span_03 <= 0.0:
        return chs_info

    adapted = []
    for ch in chs_info:
        ch_copy = dict(ch)
        loc = np.asarray(ch_copy["loc"], dtype=float).copy()
        if loc.shape[0] >= 6:
            loc[3:6] = loc[0:3]
        ch_copy["loc"] = loc
        adapted.append(ch_copy)

    print(
        "Contextual channel-loc fix applied: copied loc[0:3] -> loc[3:6] "
        "to avoid degenerate positional-encoding ranges."
    )
    return adapted

def build_model(model_name=None):
    """Instantiate the configured downstream SignalJEPA model."""
    selected_model_name = model_name or CONFIG["model_name"]
    model_cls = get_model_class(selected_model_name)

    # actual window duration
    trial_s = (WINDOW_SAMPLES / CONFIG["sfreq"] )
    if not LOG_COMPACT:
        print(f"Building model class: {selected_model_name}")
        print(f"Building model with input window duration: {trial_s:.4f} s")

    model_chs_info = CHS_INFO
    if selected_model_name == "SignalJEPA_Contextual":
        model_chs_info = adapt_chs_info_for_contextual(CHS_INFO)

    model = model_cls(
        sfreq=CONFIG["sfreq"],
        input_window_seconds=trial_s,
        chs_info=model_chs_info,
        n_outputs=TARGET_N_CLASSES,
    )
    return model

In [ ]:
# Verify once that the selected model builds without error
test_model = build_model(CONFIG["model_name"])
total_p = sum(p.numel() for p in test_model.parameters())
trainable_p = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print(f"{CONFIG['model_name']} instantiated successfully.")
print(f"  Total parameters:     {total_p:,}")
print(f"  Trainable parameters: {trainable_p:,}")
if PRINT_MODEL_SUMMARY:
    print(test_model)
del test_model

## 4.2. Load Pretrained Weights from Hugging Face

In [ ]:
def download_pretrained_state_dict(url):
    """Download Signal-JEPA pretrained weights from Hugging Face."""
    print("Initializing from 16s-60 pretrained checkpoint (Hugging Face):")
    print(f"  {url}")
    sd = torch.hub.load_state_dict_from_url(url, map_location="cpu")
    print(f"  Downloaded {len(sd)} keys")
    return sd

In [ ]:
PRETRAINED_LOAD_RULES = {
    "SignalJEPA_PreLocal": {
        "load_prefixes": ("feature_encoder.",),
        "required_checkpoint_prefixes": ("feature_encoder.",),
        "expected_missing_prefixes": ("spatial_conv.", "final_layer."),
    },
    "SignalJEPA_PostLocal": {
        "load_prefixes": ("feature_encoder.",),
        "required_checkpoint_prefixes": ("feature_encoder.",),
        "expected_missing_prefixes": ("final_layer."),
    },
    "SignalJEPA_Contextual": {
        "load_prefixes": ("feature_encoder.", "transformer."),
        "required_checkpoint_prefixes": ("feature_encoder.", "transformer."),
        "expected_missing_prefixes": ("pos_encoder.", "final_layer."),
    },
}

def _starts_with_any_prefix(key, prefixes):
    return any(key.startswith(prefix) for prefix in prefixes)

def _validate_checkpoint_prefixes(model_name, state_dict, required_prefixes):
    for prefix in required_prefixes:
        if not any(k.startswith(prefix) for k in state_dict):
            raise RuntimeError(
                f"Checkpoint is inconsistent for {model_name}: "
                f"required prefix '{prefix}' not found."
            )

def load_pretrained_weights_for_model(model, model_name, state_dict):
    """Load pretrained weights with model-specific key filtering and validation."""
    if model_name not in PRETRAINED_LOAD_RULES:
        raise ValueError(f"Unsupported model_name for loading: {model_name}")

    rules = PRETRAINED_LOAD_RULES[model_name]
    load_prefixes = rules["load_prefixes"]
    required_prefixes = rules["required_checkpoint_prefixes"]
    expected_missing_prefixes = rules["expected_missing_prefixes"]

    _validate_checkpoint_prefixes(model_name, state_dict, required_prefixes)

    filtered = {k: v for k, v in state_dict.items() if _starts_with_any_prefix(k, load_prefixes)}
    dropped = sorted([k for k in state_dict.keys() if k not in filtered])

    if len(filtered) == 0:
        raise RuntimeError(f"No keys selected for loading into {model_name}.")

    missing_keys, unexpected_keys = model.load_state_dict(filtered, strict=False)

    invalid_missing = [
        k for k in missing_keys
        if not _starts_with_any_prefix(k, expected_missing_prefixes)
    ]

    if unexpected_keys:
        raise RuntimeError(
            f"Unexpected keys while loading pretrained weights for {model_name}: "
            f"{unexpected_keys[:10]}"
        )

    if invalid_missing:
        raise RuntimeError(
            f"Unexpected missing keys for {model_name}: {invalid_missing[:10]}"
        )

    if not LOG_COMPACT:
        print("Pretrained weight loading:")
        print(f"  Model:                 {model_name}")
        print(f"  Downloaded keys:       {len(state_dict)}")
        print(f"  Keys selected:         {len(filtered)}")
        print(f"  Keys dropped:          {len(dropped)}")
        print(f"  Missing keys:          {len(missing_keys)}")
        print(f"  Unexpected keys:       {len(unexpected_keys)}")
        print(f"  Missing preview:       {list(missing_keys)[:6]}")
        print(f"  Dropped preview:       {dropped[:6]}")

    return {
        "model_name": model_name,
        "downloaded_keys": len(state_dict),
        "loaded_keys": len(filtered),
        "dropped_keys": len(dropped),
        "missing_keys": list(missing_keys),
        "unexpected_keys": list(unexpected_keys),
        "loaded_prefixes": list(load_prefixes),
        "expected_missing_prefixes": list(expected_missing_prefixes),
    }

In [ ]:
# Download once — reused for every fold
PRETRAINED_STATE_DICT = download_pretrained_state_dict(CONFIG["pretrained_url"])

## 4.3. Trainable Parameter Phases

In [ ]:
NEW_LAYER_PREFIXES = {
    "SignalJEPA_PreLocal": ("spatial_conv.", "final_layer."),
    "SignalJEPA_PostLocal": ("final_layer.",),
    "SignalJEPA_Contextual": ("pos_encoder.", "final_layer."),
}

def count_total_and_trainable_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

In [ ]:
def get_new_layer_prefixes(model_name):
    if model_name not in NEW_LAYER_PREFIXES:
        raise ValueError(f"Unsupported model_name for trainable phase logic: {model_name}")
    return NEW_LAYER_PREFIXES[model_name]

def set_trainable_params_for_phase(model, model_name, phase):
    """
    Configure trainable parameters by phase.

    phase='new' or 'warmup': only newly introduced downstream layers are trainable.
    phase='full': full model is trainable.
    """
    if phase not in ("new", "warmup", "full"):
        raise ValueError(f"Unsupported phase: {phase}")

    trainable_names = []

    if phase == "full":
        for _, param in model.named_parameters():
            param.requires_grad = True
        trainable_names = [name for name, p in model.named_parameters() if p.requires_grad]
        phase_groups = ["all_parameters"]
    else:
        downstream_prefixes = get_new_layer_prefixes(model_name)
        for _, param in model.named_parameters():
            param.requires_grad = False
        for name, param in model.named_parameters():
            if any(name.startswith(prefix) for prefix in downstream_prefixes):
                param.requires_grad = True
                trainable_names.append(name)
        phase_groups = list(downstream_prefixes)

    total, trainable = count_total_and_trainable_params(model)

    if trainable == 0:
        raise RuntimeError(
            f"No trainable parameters found for model={model_name}, phase={phase}."
        )

    return {
        "model_name": model_name,
        "phase": phase,
        "trainable_groups": phase_groups,
        "total_params": int(total),
        "trainable_params": int(trainable),
        "trainable_ratio": float(trainable / total),
        "trainable_names": trainable_names,
    }

def describe_trainable_params(summary, max_names=10):
    print(f"      Trainable groups: {summary['trainable_groups']}")
    print(
        "      Trainable params: "
        f"{summary['trainable_params']:,} / {summary['total_params']:,} "
        f"({100.0 * summary['trainable_ratio']:.1f}%)"
    )
    if PRINT_FREEZE_DETAILS:
        for name in summary["trainable_names"][:max_names]:
            print(f"        - {name}")
        if len(summary["trainable_names"]) > max_names:
            print(f"        - ... ({len(summary['trainable_names']) - max_names} more)")

# 5. Training

## 5.1. EEGClassifier Fold Runner

Paper-aligned fine-tuning semantics used below:

- `new`: train only newly introduced downstream layers; pretrained components remain frozen.
- `full`: run a warm-up where only new downstream layers are trainable, then unfreeze the full downstream model and continue training.
- Default warm-up is `CONFIG["warmup_epochs"] = 10` epochs, matching the S-JEPA paper setup.

In [ ]:
def build_classifier(model, valid_set, callbacks, max_epochs, fold_seed=None, warm_start=False):
    train_generator = None
    if fold_seed is not None:
        train_generator = torch.Generator()
        train_generator.manual_seed(fold_seed)

    clf_kwargs = {
        "batch_size": CONFIG["batch_size"],
        "max_epochs": int(max_epochs),
        "device": DEVICE,
        "callbacks": callbacks,
        "train_split": predefined_split(valid_set),
        "classes": range(TARGET_N_CLASSES),
        "iterator_train__shuffle": True,
        "iterator_train__num_workers": 0,
        "iterator_valid__num_workers": 0,
        "optimizer": torch.optim.Adam,
        "warm_start": warm_start,
    }

    if CONFIG["learning_rate"] is not None:
        clf_kwargs["lr"] = CONFIG["learning_rate"]

    if train_generator is not None:
        clf_kwargs["iterator_train__generator"] = train_generator

    return EEGClassifier(model, **clf_kwargs)

def run_one_batch_finite_sanity_check(model, train_set, model_name):
    """Fail fast if one-batch forward/loss contains NaN/Inf."""
    if len(train_set) == 0:
        raise RuntimeError(f"{model_name}: train_set is empty during sanity check.")

    batch_size = int(min(CONFIG["batch_size"], len(train_set)))
    sanity_loader = torch.utils.data.DataLoader(
        train_set,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    batch = next(iter(sanity_loader))

    if not isinstance(batch, (tuple, list)) or len(batch) < 2:
        raise RuntimeError(f"{model_name}: expected batch with at least (X, y), got type={type(batch)}")

    x_batch = batch[0]
    y_batch = batch[1]
    x_batch = torch.as_tensor(x_batch).float().to(DEVICE)
    y_batch = torch.as_tensor(y_batch).long().to(DEVICE)

    was_training = model.training
    model = model.to(DEVICE)
    model.eval()
    with torch.no_grad():
        logits = model(x_batch)
        if not torch.isfinite(logits).all():
            raise RuntimeError(
                f"{model_name}: non-finite logits detected in one-batch sanity check."
            )

        loss = torch.nn.functional.cross_entropy(logits, y_batch)
        if not torch.isfinite(loss):
            raise RuntimeError(
                f"{model_name}: non-finite loss detected in one-batch sanity check."
            )

    if was_training:
        model.train()

    print(
        f"    Sanity check passed: finite logits/loss on one training batch for {model_name}."
    )

def run_single_fold(fold_id, subject_id, subject_dataset, idx_train, idx_val, idx_test):
    """Train and evaluate one fold using EEGClassifier on dataset subsets."""
    if CONFIG["set_seed"]:
        fold_seed = BASE_SEED + int(subject_id) * 100 + int(fold_id) # type: ignore
        seed_everything(fold_seed)
    else:
        fold_seed = None

    train_set = Subset(subject_dataset, idx_train.tolist())
    valid_set = Subset(subject_dataset, idx_val.tolist())
    test_set = Subset(subject_dataset, idx_test.tolist())

    y_all = get_targets_from_dataset(subject_dataset)
    y_train = y_all[idx_train]
    y_val = y_all[idx_val]
    y_test = y_all[idx_test]

    train_counts = np.bincount(y_train, minlength=TARGET_N_CLASSES)
    val_counts = np.bincount(y_val, minlength=TARGET_N_CLASSES)
    test_counts = np.bincount(y_test, minlength=TARGET_N_CLASSES)

    if PRINT_FOLD_CLASS_COUNTS:
        print(f"    Train class counts: {train_counts.tolist()}")
        print(f"    Val class counts:   {val_counts.tolist()}")
        print(f"    Test class counts:  {test_counts.tolist()}")

    model_name = CONFIG["model_name"]
    strategy = CONFIG["strategy"]
    warmup_epochs = int(CONFIG["warmup_epochs"])

    model = build_model(model_name)
    pretrained_load_summary = load_pretrained_weights_for_model(model, model_name, PRETRAINED_STATE_DICT)

    print(f"    Downstream model:   {model_name}")
    print(f"    Fine-tune strategy: {strategy}")
    print(f"    Warm-up active:     {strategy == 'full'}")
    print(f"    Warm-up epochs:     {warmup_epochs}")
    print(f"    Missing keys:       {pretrained_load_summary['missing_keys'][:10]}")

    if (
        model_name == "SignalJEPA_Contextual"
        and any(k.startswith("pos_encoder.") for k in pretrained_load_summary["missing_keys"])
    ):
        print(
            "    Contextual note: missing pos_encoder keys are expected for this checkpoint "
            "configuration and will be trained as downstream-added parameters."
        )

    if strategy == "new":
        phase_1_summary = set_trainable_params_for_phase(model, model_name, "new")
        print("    Phase 1 (new):")
        describe_trainable_params(phase_1_summary)

        run_one_batch_finite_sanity_check(model, train_set, model_name)

        callbacks = [
            (
                "early_stopping",
                EarlyStopping(
                    monitor="valid_loss",
                    patience=CONFIG["early_stopping_patience"],
                    lower_is_better=True,
                    load_best=True,
                ),
            ),
        ]

        clf = build_classifier(
            model=model,
            valid_set=valid_set,
            callbacks=callbacks,
            max_epochs=int(CONFIG["n_epochs"]),
            fold_seed=fold_seed,
            warm_start=False,
        )

        phase_summaries = {
            "phase_1": phase_1_summary,
            "phase_2": None,
        }
        clf.fit(train_set, y=None)

    elif strategy == "full":
        if warmup_epochs < 1:
            raise ValueError("For strategy='full', warmup_epochs must be >= 1.")

        phase_1_summary = set_trainable_params_for_phase(model, model_name, "warmup")
        print("    Phase 1 (warmup):")
        describe_trainable_params(phase_1_summary)

        run_one_batch_finite_sanity_check(model, train_set, model_name)

        clf = build_classifier(
            model=model,
            valid_set=valid_set,
            callbacks=[],
            max_epochs=warmup_epochs,
            fold_seed=fold_seed,
            warm_start=True,
        )

        clf.fit(train_set, y=None)

        phase_2_summary = set_trainable_params_for_phase(clf.module_, model_name, "full")
        print("    Phase 2 (full):")
        describe_trainable_params(phase_2_summary)

        # Important: refresh optimizer so newly unfrozen parameters are trainable.
        clf.initialize_optimizer()

        remaining_epochs = int(CONFIG["n_epochs"]) - warmup_epochs
        if remaining_epochs < 1:
            raise ValueError(
                "CONFIG['n_epochs'] must be greater than CONFIG['warmup_epochs'] when strategy='full'."
            )

        callbacks = [
            (
                "early_stopping",
                EarlyStopping(
                    monitor="valid_loss",
                    patience=CONFIG["early_stopping_patience"],
                    lower_is_better=True,
                    load_best=True,
                ),
            ),
        ]

        clf.set_params(callbacks=callbacks, max_epochs=remaining_epochs)
        clf.fit(train_set, y=None)

        phase_summaries = {
            "phase_1": phase_1_summary,
            "phase_2": phase_2_summary,
        }
    else:
        raise ValueError(f"Unsupported strategy: {strategy}. Use 'new' or 'full'.")

    y_pred = clf.predict(test_set)

    stopped_epoch = int(clf.history[-1]["epoch"]) if len(clf.history) > 0 else 0 # type: ignore
    valid_loss_curve = [
        (int(row["epoch"]), float(row["valid_loss"]))
        for row in clf.history
        if "valid_loss" in row
    ]

    if valid_loss_curve:
        best_epoch, best_valid_loss = min(valid_loss_curve, key=lambda x: x[1])  # lower is better
    else:
        best_epoch, best_valid_loss = None, None

    acc = float(accuracy_score(y_test, y_pred))
    bal_acc = float(balanced_accuracy_score(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred, labels=np.arange(TARGET_N_CLASSES)).tolist()
    pred_hist = np.bincount(y_pred, minlength=TARGET_N_CLASSES).tolist()

    n_folds = CONFIG["cv_folds"]
    val_loss_str = f"{best_valid_loss:.4f}" if best_valid_loss is not None else "N/A"
    print(f"  Fold {fold_id}/{n_folds} | best_epoch={best_epoch} | stop={stopped_epoch} | val_loss={val_loss_str} | acc={acc:.4f} | bal_acc={bal_acc:.4f}")

    return {
        "subject_id": str(subject_id),
        "fold_id": fold_id,
        "model_name": model_name,
        "strategy": strategy,
        "warmup_epochs": warmup_epochs,
        "n_train": len(train_set),
        "n_val": len(valid_set),
        "n_test": len(test_set),
        "train_class_counts": train_counts.tolist(),
        "val_class_counts": val_counts.tolist(),
        "test_class_counts": test_counts.tolist(),
        "pretrained_load": pretrained_load_summary,
        "phase_1_trainable_groups": phase_summaries["phase_1"]["trainable_groups"],
        "phase_1_trainable_params": phase_summaries["phase_1"]["trainable_params"],
        "phase_2_trainable_groups": None if phase_summaries["phase_2"] is None else phase_summaries["phase_2"]["trainable_groups"],
        "phase_2_trainable_params": None if phase_summaries["phase_2"] is None else phase_summaries["phase_2"]["trainable_params"],
        "best_epoch": best_epoch,
        "stopped_epoch": stopped_epoch,
        "best_valid_loss": best_valid_loss,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "confusion_matrix": cm,
        "prediction_histogram": pred_hist,
    }

## 5.2. Subject Cross-Validation Runner

In [ ]:
def make_fold_splits(y, n_folds, val_split, n_classes):
    """Create stratified k-fold index splits for one subject."""
    indices = np.arange(len(y))
    skf = StratifiedKFold(
        n_splits=n_folds,
        shuffle=True,
        random_state=BASE_SEED if CONFIG["set_seed"] else None,
    )
    folds = []

    for fold_id, (tv_idx, test_idx) in enumerate(skf.split(indices, y), start=1):
        y_tv = y[tv_idx]
        y_test_f = y[test_idx]

        split_seed = BASE_SEED + int(fold_id) if CONFIG["set_seed"] else None # type: ignore
        tr_local_idx, val_local_idx = train_test_split(
            np.arange(len(tv_idx)),
            test_size=val_split,
            stratify=y_tv,
            random_state=split_seed,
        )

        train_idx = tv_idx[tr_local_idx]
        val_idx = tv_idx[val_local_idx]

        if PRINT_FOLD_CLASS_COUNTS:
            train_counts = np.bincount(y[train_idx], minlength=n_classes)
            val_counts = np.bincount(y[val_idx], minlength=n_classes)
            test_counts = np.bincount(y_test_f, minlength=n_classes)
            print(f"  Fold {fold_id} class counts: train={train_counts.tolist()} val={val_counts.tolist()} test={test_counts.tolist()}")

        folds.append(
            {
                "fold_id": fold_id,
                "idx_train": train_idx,
                "idx_val": val_idx,
                "idx_test": test_idx,
            }
        )

    return folds


In [ ]:
def run_subject_cv(subject_id, subject_dataset, n_classes, cv_folds, val_split):
    """Run within-subject stratified CV for one subject."""
    y = get_targets_from_dataset(subject_dataset)
    counts = np.bincount(y, minlength=n_classes)
    print(f"\nSubject {subject_id}: {len(subject_dataset)} windows | class_counts={counts.tolist()}")

    folds = make_fold_splits(
        y,
        n_folds=cv_folds,
        val_split=val_split,
        n_classes=n_classes,
    )
    results = []

    for fold in folds:
        result = run_single_fold(
            fold["fold_id"],
            subject_id,
            subject_dataset,
            fold["idx_train"],
            fold["idx_val"],
            fold["idx_test"],
        )
        results.append(result)

    mean_acc = float(np.mean([r["accuracy"] for r in results]))
    mean_bal_acc = float(np.mean([r["balanced_accuracy"] for r in results]))
    std_acc = float(np.std([r["accuracy"] for r in results]))
    std_bal_acc = float(np.std([r["balanced_accuracy"] for r in results]))
    print(f"  Subject {subject_id} summary: acc={mean_acc:.4f}±{std_acc:.4f}  bal_acc={mean_bal_acc:.4f}±{std_bal_acc:.4f}")

    return results


## 5.3. Run All Subjects

In [ ]:
print("=" * 70)
print("STARTING CROSS-VALIDATION")
print("=" * 70)
print(f"Subjects:      {list(SUBJECT_WINDOWS.keys())}")
print(f"Paradigm:      {CONFIG['paradigm']}")
print(f"Model:         {CONFIG['model_name']}")
print(f"Strategy:      {CONFIG['strategy']}")
print(f"Warm-up epochs:{CONFIG['warmup_epochs']}")
print(f"CV folds:      {CONFIG['cv_folds']}")
print(f"Batch size:    {CONFIG['batch_size']}")
print(f"Max epochs:    {CONFIG['n_epochs']}")
print(f"Device:        {DEVICE}")
print(f"Seed control:  {CONFIG['set_seed']}")
print(f"Base seed:     {BASE_SEED}")
print("=" * 70)

FOLD_RESULTS = []

for sid, subject_ds in SUBJECT_WINDOWS.items():
    fold_results = run_subject_cv(
        sid,
        subject_ds,
        TARGET_N_CLASSES,
        CONFIG["cv_folds"],
        CONFIG["val_split"],
    )
    FOLD_RESULTS.extend(fold_results)

print(f"\nTotal folds completed: {len(FOLD_RESULTS)}")

# 6. Results

## 6.1. Aggregate Metrics

In [ ]:
# Per-subject aggregation
SUBJECT_METRICS = {}

for result in FOLD_RESULTS:
    sid = result["subject_id"]
    if sid not in SUBJECT_METRICS:
        SUBJECT_METRICS[sid] = {"accuracies": [], "balanced_accuracies": []}
    SUBJECT_METRICS[sid]["accuracies"].append(result["accuracy"])
    SUBJECT_METRICS[sid]["balanced_accuracies"].append(result["balanced_accuracy"])

for sid, m in SUBJECT_METRICS.items():
    m["mean_accuracy"] = float(np.mean(m["accuracies"]))
    m["std_accuracy"] = float(np.std(m["accuracies"]))
    m["mean_balanced_accuracy"] = float(np.mean(m["balanced_accuracies"]))
    m["std_balanced_accuracy"] = float(np.std(m["balanced_accuracies"]))

# Global aggregation
all_accs = [r["accuracy"] for r in FOLD_RESULTS]
all_bal_accs = [r["balanced_accuracy"] for r in FOLD_RESULTS]

GLOBAL_METRICS = {
    "mean_accuracy": float(np.mean(all_accs)),
    "std_accuracy": float(np.std(all_accs)),
    "mean_balanced_accuracy": float(np.mean(all_bal_accs)),
    "std_balanced_accuracy": float(np.std(all_bal_accs)),
    "n_subjects": len(SUBJECT_METRICS),
    "n_folds_total": len(FOLD_RESULTS),
}

print("=" * 70)
print("AGGREGATED RESULTS")
print("=" * 70)
for sid, m in sorted(SUBJECT_METRICS.items()):
    print(f"  Subject {sid}:  acc={m['mean_accuracy']:.4f}±{m['std_accuracy']:.4f}  "
          f"bal_acc={m['mean_balanced_accuracy']:.4f}±{m['std_balanced_accuracy']:.4f}")
print("-" * 70)
print(f"  OVERALL:      acc={GLOBAL_METRICS['mean_accuracy']:.4f}±{GLOBAL_METRICS['std_accuracy']:.4f}  "
      f"bal_acc={GLOBAL_METRICS['mean_balanced_accuracy']:.4f}±{GLOBAL_METRICS['std_balanced_accuracy']:.4f}")
print("=" * 70)

## 6.2. Save Outputs

In [ ]:
cv_results_path = ARTIFACT_DIR / "cv_results.json"
with open(cv_results_path, 'w') as f:
    json.dump(FOLD_RESULTS, f, indent=2)
print(f"CV results saved to:      {cv_results_path}")

subject_metrics_path = ARTIFACT_DIR / "subject_metrics.json"
with open(subject_metrics_path, 'w') as f:
    json.dump(SUBJECT_METRICS, f, indent=2)
print(f"Subject metrics saved to: {subject_metrics_path}")

global_metrics_path = ARTIFACT_DIR / "global_metrics.json"
with open(global_metrics_path, 'w') as f:
    json.dump(GLOBAL_METRICS, f, indent=2)
print(f"Global metrics saved to:  {global_metrics_path}")

run_metadata = {
    "run_id": RUN_ID,
    "artifact_dir": str(ARTIFACT_DIR),
    "paradigm": CONFIG["paradigm"],
    "subjects": [str(s) for s in SUBJECT_TRIALS.keys()],
    "model_name": CONFIG["model_name"],
    "strategy": CONFIG["strategy"],
    "warmup_epochs": int(CONFIG["warmup_epochs"]),
    "pretrained_url": CONFIG["pretrained_url"],
    "global_metrics": GLOBAL_METRICS,
}

run_metadata_path = ARTIFACT_DIR / "run_metadata.json"
with open(run_metadata_path, 'w') as f:
    json.dump(run_metadata, f, indent=2)
print(f"Run metadata saved to:    {run_metadata_path}")

print(f"\nAll artifacts in: {ARTIFACT_DIR}")

## 6.3. Final Summary

In [ ]:
print("\n" + "=" * 70)
print("EXPERIMENT SUMMARY")
print("=" * 70)
print(f"Run ID:    {RUN_ID}")
print(f"Artifacts: {ARTIFACT_DIR}")
print()
print("Configuration:")
print(f"  Subjects:          {list(SUBJECT_TRIALS.keys())}")
print(f"  Paradigm:          {CONFIG['paradigm']}")
print(f"  Model:             {CONFIG['model_name']}")
print(f"  Strategy:          {CONFIG['strategy']}")
print(f"  Warm-up epochs:    {CONFIG['warmup_epochs']}")
print(f"  Bandpass:          {CONFIG['bandpass_low']}-{CONFIG['bandpass_high']} Hz")
print(f"  Resample:          {CONFIG['sfreq']} Hz")
print(f"  CV folds:          {CONFIG['cv_folds']}")
print(f"  Learning rate:     {CONFIG['learning_rate']}")
print(f"  Batch size:        {CONFIG['batch_size']}")
print(f"  Max epochs:        {CONFIG['n_epochs']}")
print()
print("Results:")
print(f"  Mean Accuracy:          {GLOBAL_METRICS['mean_accuracy']:.4f} ± {GLOBAL_METRICS['std_accuracy']:.4f}")
print(f"  Mean Balanced Accuracy: {GLOBAL_METRICS['mean_balanced_accuracy']:.4f} ± {GLOBAL_METRICS['std_balanced_accuracy']:.4f}")